# モデル・質問・Probeの確認
0.5B/1.5B、命名/自由連想。評価文は前回評価済みの校正集合。採用基準・限界はPLAN.md。GPU T4で全セル実行。

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install transformers==4.57.6 scikit-learn scipy pandas numpy


In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/GraduationThesis-model-check')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ryouy/GraduationThesis.git',str(REPO)],check=True)
OUT=Path('/content/model-check-20261103')
OUT.mkdir(exist_ok=True)
for size in ['0.5B','1.5B']:
    proc=subprocess.Popen([sys.executable,'-u','1103_model_check/run_check.py','--size',size,'--out',str(OUT)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with (OUT/(size+'.log')).open('w') as log:
        for line in proc.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
    assert proc.wait()==0,size
subprocess.run([sys.executable,'1103_model_check/verify.py',str(OUT)],cwd=REPO,check=True)


In [ ]:
# Post-hoc diagnostic, kept separate from prespecified calibration
for size in ['0.5B','1.5B']:
    r=subprocess.run([sys.executable,'-u','1103_model_check/run_sanity.py',str(OUT),size],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    (OUT/(size+'-sanity.log')).write_text(r.stdout)
    print(r.stdout)
    assert r.returncode==0


In [ ]:
print('MODEL_CHECK_EXPORT_START')
print(json.dumps({str(p.relative_to(OUT)):p.read_text() for p in sorted(OUT.rglob('*')) if p.is_file()}))
print('MODEL_CHECK_EXPORT_END')
